In [1]:
import multiprocessing as mp
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import cmcrameri.cm as cmc
import h5py
import hdf5plugin  # noqa: F401 -- Register compression filters for raw HDF5 data.
import matplotlib.font_manager as fm
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

%matplotlib inline

## Atlas

### Preprocessing and layout

This notebook reproduces the non-live stitching logic from 20260225 scans 226–475. Each 2849 × 2835 prediction is cropped with `[17:-32, 17:-18]` to 2800 × 2800, then mean-binned over non-overlapping 5 × 5 blocks to 560 × 560. No additional binning or intensity offset is applied. The 250 scans are placed in the original 10 × 25 atlas ordering.

In [2]:
ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "ptycho_fm").is_dir()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate the repository root")

ATLAS_DIR = Path("/mnt/localdisk/scratch/aileenluo/results/atlas_run164")
RAW_DATA_DIR = Path("/mnt/localdisk/scratch/aileenluo/raw/S26_20260225")
OUTPUT_DIR = ROOT / "workspace" / "paper"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCAN_NUMBERS = tuple(range(226, 476))
RAW_TILE_SHAPE = (2849, 2835)
CROP = (slice(17, -32), slice(17, -18))
CROPPED_TILE_SHAPE = (2800, 2800)
BIN_FACTOR = 5
BINNED_TILE_SHAPE = (560, 560)
ATLAS_GRID_SHAPE = (10, 25)
ATLAS_SHAPE = (
    ATLAS_GRID_SHAPE[0] * BINNED_TILE_SHAPE[0],
    ATLAS_GRID_SHAPE[1] * BINNED_TILE_SHAPE[1],
)

OUTPUT_DPI = 300
FULL_FIGSIZE = (ATLAS_SHAPE[1] / OUTPUT_DPI, ATLAS_SHAPE[0] / OUTPUT_DPI)
PREVIEW_FIGSIZE = (14, 5.6)
SCALEBAR_PIXELS = 1495

RAW_DATASET_PATH = "/entry/data/data"
DETECTOR_XMIN, DETECTOR_XMAX = 386, 641
DETECTOR_YMIN, DETECTOR_YMAX = 678, 933
TRANSMISSION_TILE_SHAPE = (201, 200)
TRANSMISSION_ATLAS_SHAPE = (
    ATLAS_GRID_SHAPE[0] * TRANSMISSION_TILE_SHAPE[0],
    ATLAS_GRID_SHAPE[1] * TRANSMISSION_TILE_SHAPE[1],
)
TRANSMISSION_SCALE_FACTOR = TRANSMISSION_TILE_SHAPE[1] / BINNED_TILE_SHAPE[1]
TRANSMISSION_SCALEBAR_PIXELS = round(SCALEBAR_PIXELS * TRANSMISSION_SCALE_FACTOR)
TRANSMISSION_SCALEBAR_HEIGHT = round(50 * TRANSMISSION_SCALE_FACTOR)
RAW_BATCH_SIZE = 512
MAX_RAW_WORKERS = min(16, os.cpu_count() or 1)
TRANSMISSION_CACHE = ATLAS_DIR / "transmission_atlas.npy"

# Independent switches allow any atlas to be regenerated without touching the others.
RENDER_PHASE_ATLAS = True
RENDER_AMPLITUDE_ATLAS = False
RENDER_TRANSMISSION_ATLAS = False

if not ATLAS_DIR.is_dir():
    raise FileNotFoundError(ATLAS_DIR)
if not RAW_DATA_DIR.is_dir():
    raise FileNotFoundError(RAW_DATA_DIR)

print(f"Scans: {SCAN_NUMBERS[0]}–{SCAN_NUMBERS[-1]} ({len(SCAN_NUMBERS)} total)")
print(f"Cropped tile: {CROPPED_TILE_SHAPE}; 5×5-binned tile: {BINNED_TILE_SHAPE}")
print(f"Prediction atlas: {ATLAS_SHAPE[1]:,} × {ATLAS_SHAPE[0]:,} pixels")
print(
    f"Transmission atlas (unbinned): {TRANSMISSION_ATLAS_SHAPE[1]:,} × "
    f"{TRANSMISSION_ATLAS_SHAPE[0]:,} pixels"
)
print(
    "Inclusive detector crop: "
    f"x={DETECTOR_XMIN}:{DETECTOR_XMAX}, y={DETECTOR_YMIN}:{DETECTOR_YMAX}"
)
print(f"Full-resolution figure size at {OUTPUT_DPI} dpi: {FULL_FIGSIZE[0]:.2f} × {FULL_FIGSIZE[1]:.2f} inches")

Scans: 226–475 (250 total)
Cropped tile: (2800, 2800); 5×5-binned tile: (560, 560)
Prediction atlas: 14,000 × 5,600 pixels
Transmission atlas (unbinned): 5,000 × 2,010 pixels
Inclusive detector crop: x=386:641, y=678:933
Full-resolution figure size at 300 dpi: 46.67 × 18.67 inches


In [3]:
def scan_file(modality, scan_number):
    """Return the unique amplitude or phase prediction for one scan."""
    matches = sorted(
        ATLAS_DIR.glob(
            f"pred_{modality}_object_scan_{scan_number}_*_crop104.npy"
        )
    )
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one {modality!r} file for scan {scan_number}, found {len(matches)}"
        )
    return matches[0]


def crop_and_bin(path):
    """Apply the exact crop and 5×5 mean binning used by LiveAtlas.py."""
    raw = np.load(path, mmap_mode="r")
    if raw.shape != RAW_TILE_SHAPE:
        raise ValueError(f"Unexpected shape {raw.shape} for {path.name}")
    cropped = np.asarray(raw[CROP], dtype=np.float32)
    if cropped.shape != CROPPED_TILE_SHAPE:
        raise ValueError(f"Unexpected cropped shape {cropped.shape} for {path.name}")
    return cropped.reshape(560, BIN_FACTOR, 560, BIN_FACTOR).mean(axis=(1, 3))


def atlas_position(sequence_index):
    """Map scan order to the original 10-row × 25-column LiveAtlas layout."""
    block = sequence_index // 25
    within_block = sequence_index % 25
    row = (1 - block // 5) * 5 + within_block // 5
    column = (4 - block % 5) * 5 + within_block % 5
    return row, column


def build_atlas(modality):
    """Load, preprocess, and stitch all 250 scans for one modality."""
    atlas = np.empty(ATLAS_SHAPE, dtype=np.float32)
    tile_height, tile_width = BINNED_TILE_SHAPE

    for sequence_index, scan_number in enumerate(SCAN_NUMBERS):
        tile = crop_and_bin(scan_file(modality, scan_number))
        row, column = atlas_position(sequence_index)
        row_slice = slice(row * tile_height, (row + 1) * tile_height)
        column_slice = slice(column * tile_width, (column + 1) * tile_width)
        atlas[row_slice, column_slice] = tile

        if (sequence_index + 1) % 25 == 0:
            print(f"{modality}: stitched {sequence_index + 1}/{len(SCAN_NUMBERS)} scans")

    if not np.isfinite(atlas).all():
        raise ValueError(f"The {modality} atlas contains non-finite values")
    return atlas


def add_scalebar(
    axis,
    font_size,
    scalebar_pixels=SCALEBAR_PIXELS,
    scalebar_height=50,
):
    scalebar = AnchoredSizeBar(
        axis.transData,
        scalebar_pixels,
        "50 µm",
        "lower right",
        pad=0.5,
        color="#E8E8E8",
        frameon=False,
        size_vertical=scalebar_height,
        fontproperties=fm.FontProperties(size=font_size),
    )
    axis.add_artist(scalebar)


def add_scan_rectangle(axis, scan_number, tile_shape):
    """Outline one scan at its pixel boundaries in the original atlas ordering."""
    row, column = atlas_position(SCAN_NUMBERS.index(scan_number))
    tile_height, tile_width = tile_shape
    axis.add_patch(
        plt.Rectangle(
            (column * tile_width - 0.5, row * tile_height - 0.5),
            tile_width,
            tile_height,
            fill=False,
            edgecolor="#0072B2",
            linewidth=1.5,
        )
    )


def render_atlas(
    atlas,
    title,
    cmap,
    output_path,
    scalebar_pixels=SCALEBAR_PIXELS,
    scalebar_height=50,
    highlight_scan=None,
):
    """Save every atlas pixel at 300 dpi and show a smaller inline preview."""
    mean = float(atlas.mean())
    standard_deviation = float(atlas.std())
    norm = Normalize(vmin=mean - 3 * standard_deviation, vmax=mean + 3 * standard_deviation)

    tile_shape = (
        atlas.shape[0] // ATLAS_GRID_SHAPE[0],
        atlas.shape[1] // ATLAS_GRID_SHAPE[1],
    )

    full_figsize = (atlas.shape[1] / OUTPUT_DPI, atlas.shape[0] / OUTPUT_DPI)
    full_font_size = 72 * atlas.shape[1] / ATLAS_SHAPE[1]
    full_figure = plt.figure(figsize=full_figsize, dpi=OUTPUT_DPI, frameon=False)
    full_axis = full_figure.add_axes([0, 0, 1, 1])
    full_axis.imshow(
        atlas,
        origin="lower",
        aspect="equal",
        interpolation="nearest",
        cmap=cmap,
        norm=norm,
    )
    full_axis.set_axis_off()
    if highlight_scan is not None:
        add_scan_rectangle(full_axis, highlight_scan, tile_shape)
    add_scalebar(
        full_axis,
        font_size=full_font_size,
        scalebar_pixels=scalebar_pixels,
        scalebar_height=scalebar_height,
    )
    full_figure.savefig(
        output_path,
        dpi=OUTPUT_DPI,
        transparent=True,
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.close(full_figure)

    preview_figure, preview_axis = plt.subplots(
        figsize=PREVIEW_FIGSIZE, dpi=100, layout="constrained"
    )
    preview_axis.imshow(
        atlas,
        origin="lower",
        aspect="equal",
        interpolation="nearest",
        cmap=cmap,
        norm=norm,
    )
    preview_axis.set_title(title)
    preview_axis.set_axis_off()
    if highlight_scan is not None:
        add_scan_rectangle(preview_axis, highlight_scan, tile_shape)
    add_scalebar(
        preview_axis,
        font_size=14,
        scalebar_pixels=scalebar_pixels,
        scalebar_height=scalebar_height,
    )
    plt.show()
    plt.close(preview_figure)

    print(f"Mean ± std: {mean:.6g} ± {standard_deviation:.6g}")
    print(f"Display range: [{norm.vmin:.6g}, {norm.vmax:.6g}]")
    print(
        f"Wrote {output_path.relative_to(ROOT)} at "
        f"{atlas.shape[1]:,} × {atlas.shape[0]:,} pixels"
    )

### Predicted phase

In [ ]:
if RENDER_PHASE_ATLAS:
    phase_atlas = build_atlas("ph")
    render_atlas(
        phase_atlas,
        title="Predicted phase",
        cmap=cmc.grayC,
        output_path=OUTPUT_DIR / "atlas_phase.svg",
        highlight_scan=460,
    )
    del phase_atlas
else:
    print("Skipping predicted phase atlas rendering")

### Predicted amplitude

In [ ]:
if RENDER_AMPLITUDE_ATLAS:
    amplitude_atlas = build_atlas("amp")
    render_atlas(
        amplitude_atlas,
        title="Predicted amplitude",
        cmap=cmc.oslo,
        output_path=OUTPUT_DIR / "atlas_amplitude.svg",
    )
    del amplitude_atlas
else:
    print("Skipping predicted amplitude atlas rendering")

### Transmission contrast

For each raw scan, the detector region `x=386:641`, `y=678:933` (inclusive) is loaded in bounded batches. Each cropped batch is converted from `uint16` to `int16`, negative hot-pixel values are set to zero, and only then is it summed over detector pixels for every frame. The 40,200 frame sums are placed on the native 201 × 200 raster and each scan tile stitched in the same 10 × 25 scan order as the prediction atlases, with no spatial cropping or binning.

In [ ]:
def raw_scan_file(scan_number):
    """Return the unique raw HDF5 file for one scan."""
    matches = sorted(RAW_DATA_DIR.glob(f"scan_{scan_number}_*.h5"))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one raw HDF5 file for scan {scan_number}, found {len(matches)}"
        )
    return matches[0]


def transmission_tile_from_raw(task):
    """Sum the inclusive detector crop and form one native serpentine scan map."""
    scan_number, raw_path = task
    frame_count = int(np.prod(TRANSMISSION_TILE_SHAPE))
    frame_sums = np.empty(frame_count, dtype=np.float64)

    with h5py.File(raw_path, "r") as handle:
        if RAW_DATASET_PATH not in handle:
            raise KeyError(f"{raw_path} does not contain {RAW_DATASET_PATH}")
        detector_data = handle[RAW_DATASET_PATH]
        if detector_data.shape[0] != frame_count:
            raise ValueError(
                f"Scan {scan_number} has {detector_data.shape[0]} frames; "
                f"expected {frame_count} for a {TRANSMISSION_TILE_SHAPE} map"
            )
        if DETECTOR_YMAX >= detector_data.shape[1] or DETECTOR_XMAX >= detector_data.shape[2]:
            raise ValueError(
                f"Detector crop exceeds raw shape {detector_data.shape[1:]} "
                f"for scan {scan_number}"
            )

        for start in range(0, frame_count, RAW_BATCH_SIZE):
            stop = min(start + RAW_BATCH_SIZE, frame_count)
            frames = detector_data[
                start:stop,
                DETECTOR_YMIN : DETECTOR_YMAX + 1,
                DETECTOR_XMIN : DETECTOR_XMAX + 1,
            ].astype(np.int16, copy=False)
            frames[frames < 0] = 0
            frame_sums[start:stop] = frames.sum(
                axis=(1, 2), dtype=np.int64
            )

    tile = frame_sums.reshape(TRANSMISSION_TILE_SHAPE)
    tile[1::2] = tile[1::2, ::-1].copy()
    return scan_number, tile.astype(np.float32)


def build_transmission_atlas(max_workers=MAX_RAW_WORKERS):
    """Build the unbinned transmission atlas from all 250 raw scans."""
    tasks = [(scan_number, str(raw_scan_file(scan_number))) for scan_number in SCAN_NUMBERS]
    atlas = np.empty(TRANSMISSION_ATLAS_SHAPE, dtype=np.float32)
    tile_height, tile_width = TRANSMISSION_TILE_SHAPE

    context = mp.get_context("fork")
    with ProcessPoolExecutor(max_workers=max_workers, mp_context=context) as executor:
        futures = {
            executor.submit(transmission_tile_from_raw, task): task[0]
            for task in tasks
        }
        for completed_count, future in enumerate(as_completed(futures), start=1):
            scan_number, tile = future.result()
            sequence_index = scan_number - SCAN_NUMBERS[0]
            row, column = atlas_position(sequence_index)
            row_slice = slice(row * tile_height, (row + 1) * tile_height)
            column_slice = slice(column * tile_width, (column + 1) * tile_width)
            atlas[row_slice, column_slice] = tile

            if completed_count % 25 == 0 or completed_count == len(tasks):
                print(f"transmission: processed {completed_count}/{len(tasks)} scans")

    if not np.isfinite(atlas).all():
        raise ValueError("The transmission atlas contains non-finite values")
    return atlas


def load_or_build_transmission_atlas(rebuild=False):
    if TRANSMISSION_CACHE.is_file() and not rebuild:
        atlas = np.load(TRANSMISSION_CACHE)
        if atlas.shape != TRANSMISSION_ATLAS_SHAPE:
            raise ValueError(
                f"Cached transmission atlas has shape {atlas.shape}; "
                f"expected {TRANSMISSION_ATLAS_SHAPE}"
            )
        if not np.isfinite(atlas).all():
            raise ValueError("Cached transmission atlas contains non-finite values")
        print(f"Loaded cached transmission atlas from {TRANSMISSION_CACHE}")
        return atlas

    atlas = build_transmission_atlas()
    np.save(TRANSMISSION_CACHE, atlas)
    print(f"Cached transmission atlas at {TRANSMISSION_CACHE}")
    return atlas


if RENDER_TRANSMISSION_ATLAS:
    transmission_atlas = load_or_build_transmission_atlas()
    render_atlas(
        transmission_atlas,
        title="Transmission contrast",
        cmap=cmc.oslo,
        output_path=OUTPUT_DIR / "atlas_transmission.svg",
        scalebar_pixels=TRANSMISSION_SCALEBAR_PIXELS,
        scalebar_height=TRANSMISSION_SCALEBAR_HEIGHT,
    )
    del transmission_atlas
else:
    print("Skipping transmission atlas rendering")

## Scan 460: full-resolution predictions and transmission contrast

Run the cells in this section in order, independently of the atlas cells above.
The amplitude and phase arrays retain all 2849 × 2835 pixels and their original
dtype: no atlas crop, 5 × 5 mean binning, intensity offset, or normalization is
applied to the prediction data.

Transmission uses the atlas calculation on scan 460's raw diffraction frames:
select detector pixels x=386–641 and y=678–933 (inclusive), convert each batch to
`int16`, zero negative hot-pixel values, sum detector pixels, reshape the 40,200
sums to 201 × 200, and reverse alternate rows for the serpentine scan. The result
is the same unnormalized summed-intensity contrast used in the atlas, stored as
`float32`, with no spatial cropping or binning.

The resulting arrays are `scan460_amplitude`, `scan460_phase`, and
`scan460_transmission`. The final cell is an inline preview; it leaves these
full-resolution arrays unchanged.


In [2]:
def load_scan460(
    prediction_dir="/mnt/localdisk/scratch/aileenluo/results/atlas_run164",
    raw_dir="/mnt/localdisk/scratch/aileenluo/raw/S26_20260225",
    batch_size=512,
):
    """Load full-resolution predictions and calculate the atlas transmission tile."""
    from pathlib import Path

    import h5py
    import hdf5plugin as _hdf5plugin  # noqa: F401 -- Register raw HDF5 compression filters.
    import numpy as np

    def unique_file(directory, pattern):
        matches = sorted(Path(directory).glob(pattern))
        if len(matches) != 1:
            raise FileNotFoundError(
                f"Expected one file matching {pattern!r} in {directory}, found {len(matches)}"
            )
        return matches[0]

    amplitude = np.load(unique_file(prediction_dir, "pred_amp_object_scan_460_*_crop104.npy"))
    phase = np.load(unique_file(prediction_dir, "pred_ph_object_scan_460_*_crop104.npy"))
    for modality, prediction in (("amplitude", amplitude), ("phase", phase)):
        if prediction.shape != (2849, 2835):
            raise ValueError(f"Unexpected scan 460 {modality} shape: {prediction.shape}")
        if not np.isfinite(prediction).all():
            raise ValueError(f"Scan 460 {modality} contains non-finite values")
        print(f"Scan 460 {modality}: {prediction.shape}, {prediction.dtype}; full resolution")

    # Keep these settings and operations identical to the transmission atlas above.
    raw_path = unique_file(raw_dir, "scan_460_*.h5")
    dataset_path = "/entry/data/data"
    transmission_shape = (201, 200)
    frame_count = int(np.prod(transmission_shape))
    detector_xmin, detector_xmax = 386, 641
    detector_ymin, detector_ymax = 678, 933
    frame_sums = np.empty(frame_count, dtype=np.float64)
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    with h5py.File(raw_path, "r") as handle:
        if dataset_path not in handle:
            raise KeyError(f"{raw_path} does not contain {dataset_path}")
        detector_data = handle[dataset_path]
        if detector_data.ndim != 3:
            raise ValueError(f"Expected 3D detector data, got {detector_data.shape}")
        if detector_data.shape[0] != frame_count:
            raise ValueError(
                f"Scan 460 has {detector_data.shape[0]} frames; "
                f"expected {frame_count} for a {transmission_shape} map"
            )
        if detector_ymax >= detector_data.shape[1] or detector_xmax >= detector_data.shape[2]:
            raise ValueError(f"Detector crop exceeds raw shape {detector_data.shape[1:]}")

        for start in range(0, frame_count, batch_size):
            stop = min(start + batch_size, frame_count)
            frames = detector_data[
                start:stop,
                detector_ymin : detector_ymax + 1,
                detector_xmin : detector_xmax + 1,
            ].astype(np.int16, copy=False)
            frames[frames < 0] = 0
            frame_sums[start:stop] = frames.sum(axis=(1, 2), dtype=np.int64)
            if stop % (10 * batch_size) == 0 or stop == frame_count:
                print(f"Scan 460 transmission: {stop:,}/{frame_count:,} frames")

    transmission = frame_sums.reshape(transmission_shape)
    transmission[1::2] = transmission[1::2, ::-1].copy()
    transmission = transmission.astype(np.float32)
    if not np.isfinite(transmission).all():
        raise ValueError("Scan 460 transmission contains non-finite values")
    print(f"Scan 460 transmission: {transmission.shape}; native scan resolution")
    return amplitude, phase, transmission


In [3]:
scan460_amplitude, scan460_phase, scan460_transmission = load_scan460()


Scan 460 amplitude: (2849, 2835), float32; full resolution
Scan 460 phase: (2849, 2835), float32; full resolution
Scan 460 transmission: 5,120/40,200 frames
Scan 460 transmission: 10,240/40,200 frames
Scan 460 transmission: 15,360/40,200 frames
Scan 460 transmission: 20,480/40,200 frames
Scan 460 transmission: 25,600/40,200 frames
Scan 460 transmission: 30,720/40,200 frames
Scan 460 transmission: 35,840/40,200 frames
Scan 460 transmission: 40,200/40,200 frames
Scan 460 transmission: (201, 200); native scan resolution


In [ ]:
def preview_scan460(amplitude, phase, transmission):
    """Preview all three modalities without changing their full-resolution data."""
    import cmcrameri.cm as cmc
    from matplotlib import pyplot as plt
    from matplotlib.colors import Normalize
    from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
    from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable

    figure, axes = plt.subplots(1, 3, figsize=(13.5, 5), dpi=300, layout="constrained")
    figure.get_layout_engine().set(wspace=0.18, w_pad=0.08)
    for axis, image, title, cmap in zip(
        axes,
        (amplitude, phase, transmission),
        ("Predicted amplitude", "Predicted phase", "Transmission contrast"),
        (cmc.oslo, cmc.grayC, cmc.oslo),
    ):
        # Match the atlas display range without changing the underlying arrays.
        mean = float(image.mean())
        std = float(image.std())
        artist = axis.imshow(
            image,
            origin="lower",
            aspect="equal",
            interpolation="nearest",
            cmap=cmap,
            norm=Normalize(vmin=mean - 3 * std, vmax=mean + 3 * std),
        )
        axis.set_title(f"Scan 460: {title}")
        axis.set_xticks([])
        axis.set_yticks([])
        divider = make_axes_locatable(axis)
        colorbar_axis = divider.append_axes("right", size="5%", pad=0.05)
        figure.colorbar(artist, cax=colorbar_axis, orientation="vertical")

    scalebar = AnchoredSizeBar(
        axes[1].transData,
        299,
        r"$2 \mu m$",
        "lower right",
        pad=0.2,
        color="#E8E8E8",
        frameon=False,
        size_vertical=5,
    )
    axes[1].add_artist(scalebar)
    figure.savefig("fig5_scan460.svg", dpi=300, transparent=True)
    plt.show()


preview_scan460(scan460_amplitude, scan460_phase, scan460_transmission)
